# Visão Computacional para Segurança do Trabalho — Pipeline Completo

Todas as fases (1-4) e a aplicação prática (relatório de conformidade) em um único notebook, na ordem de execução. Equivalente a rodar os 6 notebooks separados (`visao_computacional_01_data.ipynb` a `visao_computacional_06_report.ipynb`) em sequência, na mesma sessão do Colab — sem precisar reabrir/remontar o Drive entre eles.

Ver `docs/relatorio-tecnico.md` no repositório para os resultados completos já obtidos com este mesmo pipeline.

## Setup (uma única vez para todo o pipeline)

In [ ]:
!pip install -q ultralytics opencv-python pandas pyarrow matplotlib pillow
# --upgrade e necessario: o Colab ja vem com uma versao antiga (2.0.2) do
# pacote kaggle pre-instalada, que nao suporta o token novo nem "python -m kaggle".
!pip install -q --upgrade kaggle

from pathlib import Path

# Armazenamento persistente compartilhado entre os notebooks: monta o Google
# Drive e usa uma pasta fixa. Troque o caminho se preferir outra estrutura.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/vc-seguranca-trabalho')
except ImportError:
    # Execucao fora do Colab (teste local) - usa uma pasta local.
    PROJECT_DIR = Path('./vc-seguranca-trabalho').resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ROOT = PROJECT_DIR
print("Diretorio do projeto:", ROOT)


# Fase 1 — Dados

Baixa e prepara os dois datasets (Construction Site Safety para detecção, COCO person para segmentação), gera splits reprodutíveis e roda a análise exploratória (EDA).

Ver `docs/relatorio-tecnico.md` no repositório para os resultados já obtidos com estes mesmos passos.

### Token do Kaggle

O dataset "Construction Site Safety" é baixado do Kaggle e exige autenticação.
Gere um token em [kaggle.com/settings/api](https://www.kaggle.com/settings/api)
("Create New Token") e cole abaixo (fica só nesta sessão do Colab — não é salvo no notebook).

In [ ]:
import os

KAGGLE_TOKEN = ""  # cole aqui o token gerado em kaggle.com/settings/api

if KAGGLE_TOKEN:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "access_token").write_text(KAGGLE_TOKEN)
    print("Token do Kaggle configurado.")
else:
    print("AVISO: configure KAGGLE_TOKEN acima antes de rodar a proxima celula.")


## 1. Baixar e amostrar o dataset de detecção (Construction Site Safety)

Baixa o dataset completo do Kaggle e seleciona um subconjunto de ~350 imagens por amostragem estratificada (seed=42), garantindo presença mínima de todas as 10 classes de EPI.

In [ ]:
import random
import shutil
import subprocess
import sys
import tempfile
from collections import defaultdict
from pathlib import Path

SEED = 42
TARGET_TOTAL = 350
MIN_PER_CLASS = 15

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]

KAGGLE_DATASET = "snehilsanyal/construction-site-safety-image-dataset-roboflow"
OUTPUT_ROOT = ROOT / "data" / "raw" / "construction-site-safety"


def download_kaggle_dataset(tmp_dir: Path) -> Path:
    """Baixa (uma unica vez) o dataset do Kaggle via CLI. Requer um token
    configurado em ~/.kaggle/access_token (celula acima)."""
    css_data_dir = tmp_dir / "css-data"
    if css_data_dir.exists():
        return css_data_dir

    token_path = Path.home() / ".kaggle" / "access_token"
    if not token_path.exists():
        raise RuntimeError(
            f"Token do Kaggle nao encontrado em {token_path}. "
            "Rode a celula 'Token do Kaggle' acima antes desta."
        )

    print("Baixando dataset 'Construction Site Safety' do Kaggle (~206MB)...")
    result = subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", KAGGLE_DATASET, "-p", str(tmp_dir), "--unzip"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"Falha ao baixar dataset do Kaggle (codigo {result.returncode}). "
            "Veja a mensagem de erro acima - geralmente e token invalido/expirado."
        )
    return css_data_dir


def read_classes_in_label(label_path: Path) -> set:
    classes = set()
    if not label_path.exists():
        return classes
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cls_id = int(line.split()[0])
            classes.add(cls_id)
    return classes


def collect_pool(source_root: Path):
    pool = []
    for split in ["train", "valid", "test"]:
        images_dir = source_root / split / "images"
        labels_dir = source_root / split / "labels"
        if not images_dir.exists():
            continue
        for img_path in sorted(images_dir.iterdir()):
            if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            label_path = labels_dir / (img_path.stem + ".txt")
            classes = read_classes_in_label(label_path)
            pool.append((img_path, label_path, classes))
    return pool


random.seed(SEED)

tmp_dir = Path(tempfile.gettempdir()) / "css_kaggle_cache"
tmp_dir.mkdir(parents=True, exist_ok=True)
source_root = download_kaggle_dataset(tmp_dir)

pool = collect_pool(source_root)
print(f"Total de imagens disponiveis na fonte: {len(pool)}")

random.shuffle(pool)

selected = []
selected_paths = set()
class_counts = defaultdict(int)

# Passo 1: garantir cobertura minima de cada classe
for cls_id in range(len(CLASS_NAMES)):
    for img_path, label_path, classes in pool:
        if img_path in selected_paths:
            continue
        if cls_id in classes and class_counts[cls_id] < MIN_PER_CLASS:
            selected.append((img_path, label_path, classes))
            selected_paths.add(img_path)
            for c in classes:
                class_counts[c] += 1
        if class_counts[cls_id] >= MIN_PER_CLASS:
            break

# Passo 2: completar ate o total alvo com amostragem aleatoria (seed fixa)
for img_path, label_path, classes in pool:
    if len(selected) >= TARGET_TOTAL:
        break
    if img_path in selected_paths:
        continue
    selected.append((img_path, label_path, classes))
    selected_paths.add(img_path)
    for c in classes:
        class_counts[c] += 1

print(f"Subconjunto selecionado: {len(selected)} imagens (seed={SEED})")
print("Contagem de instancias-imagem por classe no subconjunto:")
for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {class_counts.get(cls_id, 0)} imagens")

images_out = OUTPUT_ROOT / "images"
labels_out = OUTPUT_ROOT / "labels"
images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)

copied = 0
for img_path, label_path, _ in selected:
    shutil.copy2(img_path, images_out / img_path.name)
    if label_path.exists():
        shutil.copy2(label_path, labels_out / label_path.name)
    copied += 1

print(f"Imagens+labels copiados: {copied}")
print("data.yaml (formato Ultralytics, com train/val/test) e escrito na secao 5, nao aqui.")


## 2. Baixar e amostrar o dataset de segmentação (COCO person)

Baixa as anotações COCO 2017 (val2017), filtra a categoria `person` com máscara, e baixa 300 imagens (seed=42). Fonte pública, não exige credenciais.

In [ ]:
import json
import random
import tempfile
import urllib.request
import zipfile
from pathlib import Path

SEED = 42
TARGET_IMAGES = 300
ANNOTATIONS_ZIP_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANNOTATIONS_MEMBER = "annotations/instances_val2017.json"
OUTPUT_DIR = ROOT / "data" / "raw" / "coco-person"
IMAGES_DIR = OUTPUT_DIR / "images"
COCO_IMAGE_BASE_URL = "http://images.cocodataset.org/val2017/"


def download_and_extract_annotations(tmp_dir: Path) -> Path:
    """Baixa apenas o instances_val2017.json de dentro do zip de anotacoes do
    COCO (241MB), sem precisar manter o zip inteiro em disco depois."""
    extracted_path = tmp_dir / "instances_val2017.json"
    if extracted_path.exists():
        return extracted_path

    zip_path = tmp_dir / "annotations_trainval2017.zip"
    print("Baixando anotacoes do COCO (241MB, so uma vez)...")
    urllib.request.urlretrieve(ANNOTATIONS_ZIP_URL, zip_path)

    with zipfile.ZipFile(zip_path) as zf:
        zf.extract(ANNOTATIONS_MEMBER, tmp_dir)
    (tmp_dir / ANNOTATIONS_MEMBER).rename(extracted_path)
    zip_path.unlink()
    return extracted_path


random.seed(SEED)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

tmp_dir = Path(tempfile.gettempdir()) / "coco_annotations_cache"
tmp_dir.mkdir(parents=True, exist_ok=True)
annotations_path = download_and_extract_annotations(tmp_dir)

with open(annotations_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

person_cat_id = next(c["id"] for c in coco["categories"] if c["name"] == "person")
images_by_id = {img["id"]: img for img in coco["images"]}

person_anns_by_image = {}
for ann in coco["annotations"]:
    if ann["category_id"] == person_cat_id and ann.get("iscrowd", 0) == 0:
        person_anns_by_image.setdefault(ann["image_id"], []).append(ann)

eligible_image_ids = [img_id for img_id, anns in person_anns_by_image.items() if len(anns) >= 1]
eligible_image_ids.sort()
random.shuffle(eligible_image_ids)

selected_ids = sorted(eligible_image_ids[:TARGET_IMAGES])
print(f"Imagens elegiveis (com pessoa e mascara): {len(eligible_image_ids)}")
print(f"Selecionadas (seed={SEED}): {len(selected_ids)}")

selected_images = [images_by_id[i] for i in selected_ids]
selected_annotations = [ann for i in selected_ids for ann in person_anns_by_image[i]]

subset = {
    "info": coco.get("info", {}),
    "licenses": coco.get("licenses", []),
    "categories": [c for c in coco["categories"] if c["id"] == person_cat_id],
    "images": selected_images,
    "annotations": selected_annotations,
}

subset_path = OUTPUT_DIR / "instances_person_subset.json"
with open(subset_path, "w", encoding="utf-8") as f:
    json.dump(subset, f)
print(f"Anotacoes filtradas salvas em: {subset_path}")

total_instances = len(selected_annotations)
print(f"Total de instancias de pessoa no subconjunto: {total_instances}")

downloaded = 0
for img in selected_images:
    dest = IMAGES_DIR / img["file_name"]
    if dest.exists():
        downloaded += 1
        continue
    url = COCO_IMAGE_BASE_URL + img["file_name"]
    try:
        urllib.request.urlretrieve(url, dest)
        downloaded += 1
    except Exception as e:
        print(f"Falha ao baixar {img['file_name']}: {e}")

print(f"Imagens baixadas: {downloaded}/{len(selected_images)}")


## 3. Gerar splits reprodutíveis (70/20/10)

Gera as listas de treino/validação/teste para as duas fontes, com seed=42.

In [ ]:
import random
from pathlib import Path

SEED = 42
RATIOS = {"train": 0.7, "val": 0.2, "test": 0.1}

SOURCES = {
    "construction-site-safety": ROOT / "data" / "raw" / "construction-site-safety" / "images",
    "coco-person": ROOT / "data" / "raw" / "coco-person" / "images",
}
SPLITS_ROOT = ROOT / "data" / "splits"


def split_list(files, ratios, seed):
    files = sorted(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    n = len(files)
    n_train = int(n * ratios["train"])
    n_val = int(n * ratios["val"])
    return {
        "train": files[:n_train],
        "val": files[n_train : n_train + n_val],
        "test": files[n_train + n_val :],
    }


for source_name, images_dir in SOURCES.items():
    if not images_dir.exists():
        print(f"[aviso] pasta nao encontrada, pulando: {images_dir}")
        continue
    files = [p.name for p in images_dir.iterdir() if p.is_file()]
    splits = split_list(files, RATIOS, SEED)

    out_dir = SPLITS_ROOT / source_name
    out_dir.mkdir(parents=True, exist_ok=True)
    for split_name, split_files in splits.items():
        out_path = out_dir / f"{split_name}.txt"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("\n".join(split_files) + "\n")

    print(f"{source_name}: total={len(files)} "
          f"train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])} "
          f"(seed={SEED})")


## 4. Análise exploratória (EDA)

Contagem de instâncias por classe, resolução e brilho médio (proxy de iluminação) do dataset de detecção.

In [ ]:
from collections import Counter
from pathlib import Path

from PIL import Image
import matplotlib.pyplot as plt

DATA_DIR = ROOT / "data" / "raw" / "construction-site-safety"
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
FIGURES_DIR = ROOT / "reports" / "figures"

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]


def count_instances_per_class():
    counts = Counter()
    for label_path in LABELS_DIR.glob("*.txt"):
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls_id = int(line.split()[0])
                counts[cls_id] += 1
    return counts


def sample_resolutions_and_brightness(sample_size=100):
    image_paths = sorted(IMAGES_DIR.iterdir())[:sample_size]
    resolutions = []
    brightness = []
    for p in image_paths:
        try:
            with Image.open(p) as img:
                resolutions.append(img.size)
                gray = img.convert("L")
                pixels = list(gray.getdata())
                brightness.append(sum(pixels) / len(pixels))
        except Exception as e:
            print(f"[aviso] falha ao ler {p.name}: {e}")
    return resolutions, brightness


FIGURES_DIR.mkdir(parents=True, exist_ok=True)

counts = count_instances_per_class()
total_instances = sum(counts.values())
print("Instancias por classe:")
for cls_id, name in enumerate(CLASS_NAMES):
    n = counts.get(cls_id, 0)
    pct = (n / total_instances * 100) if total_instances else 0
    print(f"  {name}: {n} ({pct:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))
values = [counts.get(i, 0) for i in range(len(CLASS_NAMES))]
ax.bar(CLASS_NAMES, values, color="#4C72B0")
ax.set_ylabel("Numero de instancias")
ax.set_title("Instancias por classe - Construction Site Safety (subconjunto)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "css_instances_per_class.png", dpi=150)
plt.show()

resolutions, brightness = sample_resolutions_and_brightness(sample_size=150)
widths = [w for w, h in resolutions]
heights = [h for w, h in resolutions]

print(f"\nResolucao (amostra de {len(resolutions)} imagens):")
if widths:
    print(f"  largura: min={min(widths)} max={max(widths)} media={sum(widths)/len(widths):.0f}")
    print(f"  altura:  min={min(heights)} max={max(heights)} media={sum(heights)/len(heights):.0f}")
if brightness:
    print(f"  brilho medio (0-255): min={min(brightness):.1f} max={max(brightness):.1f} "
          f"media={sum(brightness)/len(brightness):.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=20, color="#55A868")
axes[0].set_title("Distribuicao de largura (px)")
axes[1].hist(brightness, bins=20, color="#C44E52")
axes[1].set_title("Distribuicao de brilho medio (proxy de iluminacao)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "css_resolution_brightness.png", dpi=150)
plt.show()

max_class = max(counts, key=counts.get)
min_class = min(range(len(CLASS_NAMES)), key=lambda i: counts.get(i, 0))
ratio = counts[max_class] / max(counts.get(min_class, 1), 1)
print(f"\nDesbalanceamento: classe mais frequente = {CLASS_NAMES[max_class]} "
      f"({counts[max_class]} instancias); classe menos frequente = "
      f"{CLASS_NAMES[min_class]} ({counts.get(min_class, 0)} instancias); "
      f"razao aproximada = {ratio:.1f}x")


# Fase 2 — Detecção de EPIs

Fine-tuning do YOLOv8n para as 10 classes de EPI, a partir dos dados preparados no notebook `01_data.ipynb` (precisa ter sido executado antes, ou os dados já estarem no mesmo `PROJECT_DIR` no Google Drive).

## 5. Preparar configuração YOLO (detecção)

Gera `data.yaml` e as listas `train.txt`/`val.txt`/`test.txt` no formato Ultralytics, a partir dos splits do notebook de dados.

In [ ]:
from pathlib import Path

DATASET_DIR = ROOT / "data" / "raw" / "construction-site-safety"
SPLITS_DIR = ROOT / "data" / "splits" / "construction-site-safety"

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]

for split in ["train", "val", "test"]:
    manifest = SPLITS_DIR / f"{split}.txt"
    filenames = [
        line.strip() for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()
    ]
    out_path = DATASET_DIR / f"{split}.txt"
    images_dir = DATASET_DIR / "images"
    with open(out_path, "w", encoding="utf-8") as f:
        for name in filenames:
            f.write((images_dir / name).as_posix() + "\n")
    print(f"{split}: {len(filenames)} imagens -> {out_path}")

data_yaml = DATASET_DIR / "data.yaml"
with open(data_yaml, "w", encoding="utf-8") as f:
    f.write(f"path: {DATASET_DIR.as_posix()}\n")
    f.write("train: train.txt\n")
    f.write("val: val.txt\n")
    f.write("test: test.txt\n")
    f.write(f"nc: {len(CLASS_NAMES)}\n")
    f.write(f"names: {CLASS_NAMES}\n")

print(f"data.yaml (Ultralytics) escrito em: {data_yaml}")


## 6. Treinar o detector (YOLOv8n)

In [ ]:
from ultralytics import YOLO

DATA_YAML = ROOT / "data" / "raw" / "construction-site-safety" / "data.yaml"
PROJECT_DIR = ROOT / "models" / "detection"
RUN_NAME = "css_yolov8n_baseline"

# Hiperparametros documentados em docs/relatorio-tecnico.md (secao 3.1).
# Com GPU (Colab), este treino roda em poucos minutos - no desenvolvimento
# original (CPU), levou ~59 minutos para as mesmas 30 epocas.
HYPERPARAMS = dict(
    model="yolov8n.pt",
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer="auto",
    seed=42,
    patience=100,
)

model = YOLO(HYPERPARAMS["model"])
model.train(
    data=str(DATA_YAML),
    epochs=HYPERPARAMS["epochs"],
    imgsz=HYPERPARAMS["imgsz"],
    batch=HYPERPARAMS["batch"],
    optimizer=HYPERPARAMS["optimizer"],
    seed=HYPERPARAMS["seed"],
    patience=HYPERPARAMS["patience"],
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)
print(f"Treino concluido. Resultados em: {PROJECT_DIR / RUN_NAME}")


## Resultado esperado

mAP@0.5 ≈ 0.49 (validação) — ver `docs/relatorio-tecnico.md` seção 4.1 para os números completos obtidos no desenvolvimento original.

# Fase 3 — Segmentação de Pessoas

Converte as anotações COCO para YOLO-seg, treina o YOLOv8n-seg (classe `person`) e compara visualmente com o detector da Fase 2. Requer os dados do notebook `01_data.ipynb` e os pesos do notebook `02_detection.ipynb` (para a comparação).

## 7. Converter COCO para YOLO-seg

Converte os polígonos de anotação do COCO para o formato YOLO-seg.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

DATASET_DIR = ROOT / "data" / "raw" / "coco-person"
ANNOTATIONS_PATH = DATASET_DIR / "instances_person_subset.json"
SPLITS_DIR = ROOT / "data" / "splits" / "coco-person"
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"

CLASS_ID = 0  # unica classe: person


def polygon_to_yolo_seg(segmentation, img_w, img_h):
    """Converte uma lista de poligonos COCO (pixel) em linhas YOLO-seg
    normalizadas (0-1). Poligonos com menos de 3 pontos sao descartados."""
    lines = []
    for poly in segmentation:
        if len(poly) < 6:
            continue
        norm = []
        for i in range(0, len(poly), 2):
            x = poly[i] / img_w
            y = poly[i + 1] / img_h
            norm.append(f"{x:.6f}")
            norm.append(f"{y:.6f}")
        lines.append(f"{CLASS_ID} " + " ".join(norm))
    return lines


with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    coco = json.load(f)

images_by_id = {img["id"]: img for img in coco["images"]}
anns_by_image = defaultdict(list)
for ann in coco["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

LABELS_DIR.mkdir(parents=True, exist_ok=True)

total_labels_written = 0
total_polygons = 0
for img_id, anns in anns_by_image.items():
    img = images_by_id[img_id]
    w, h = img["width"], img["height"]
    stem = Path(img["file_name"]).stem
    label_path = LABELS_DIR / f"{stem}.txt"

    lines = []
    for ann in anns:
        lines.extend(polygon_to_yolo_seg(ann["segmentation"], w, h))

    if lines:
        with open(label_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
        total_labels_written += 1
        total_polygons += len(lines)

print(f"Labels YOLO-seg escritos: {total_labels_written} (poligonos totais: {total_polygons})")

for split in ["train", "val", "test"]:
    manifest = SPLITS_DIR / f"{split}.txt"
    filenames = [
        line.strip() for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()
    ]
    out_path = DATASET_DIR / f"{split}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        for name in filenames:
            f.write((IMAGES_DIR / name).as_posix() + "\n")
    print(f"{split}: {len(filenames)} imagens -> {out_path}")

data_yaml = DATASET_DIR / "data.yaml"
with open(data_yaml, "w", encoding="utf-8") as f:
    f.write(f"path: {DATASET_DIR.as_posix()}\n")
    f.write("train: train.txt\n")
    f.write("val: val.txt\n")
    f.write("test: test.txt\n")
    f.write("nc: 1\n")
    f.write("names: ['person']\n")

print(f"data.yaml (Ultralytics, segmentacao) escrito em: {data_yaml}")


## 8. Treinar o segmentador (YOLOv8n-seg)

In [ ]:
from ultralytics import YOLO

DATA_YAML = ROOT / "data" / "raw" / "coco-person" / "data.yaml"
PROJECT_DIR = ROOT / "models" / "segmentation"
RUN_NAME = "coco_person_yolov8n_seg_baseline"

# Mesmos hiperparametros de forma da Fase 2, para manter os dois modelos
# comparaveis (docs/relatorio-tecnico.md, secao 3.2).
HYPERPARAMS = dict(
    model="yolov8n-seg.pt",
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer="auto",
    seed=42,
    patience=100,
)

model = YOLO(HYPERPARAMS["model"])
model.train(
    data=str(DATA_YAML),
    epochs=HYPERPARAMS["epochs"],
    imgsz=HYPERPARAMS["imgsz"],
    batch=HYPERPARAMS["batch"],
    optimizer=HYPERPARAMS["optimizer"],
    seed=HYPERPARAMS["seed"],
    patience=HYPERPARAMS["patience"],
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)
print(f"Treino concluido. Resultados em: {PROJECT_DIR / RUN_NAME}")


## 9. Comparação visual caixas × máscaras

Roda os dois modelos treinados sobre as mesmas imagens do domínio de canteiro de obra (requer os pesos do detector já treinados em `02_detection.ipynb`).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
SEGMENTATION_WEIGHTS = ROOT / "models" / "segmentation" / "coco_person_yolov8n_seg_baseline" / "weights" / "best.pt"
CSS_IMAGES_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "images"
CSS_LABELS_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "labels"
FIGURES_DIR = ROOT / "reports" / "figures"

PERSON_CLASS_ID_DETECTION = 5  # indice da classe "Person" no dataset de EPIs
NUM_EXAMPLES = 2
CROPS_DIR = FIGURES_DIR / "_crops"

# NOTA: boa parte do dataset Construction Site Safety (versao exportada) e
# composta por imagens-mosaico 2x2 (4 fotos combinadas numa so) - ver
# docs/relatorio-tecnico.md. Para uma comparacao visual legivel, recortamos
# o quadrante onde a pessoa esta centrada. A escolha de imagem/quadrante e
# dinamica (nao usa nomes de arquivo fixos), pois o subconjunto de 350
# imagens da Fase 1 pode variar entre maquinas/sistemas operacionais.


def crop_quadrant(img_path, quadrant, out_path):
    img = Image.open(img_path)
    w, h = img.size
    boxes = {
        "tl": (0, 0, w // 2, h // 2),
        "tr": (w // 2, 0, w, h // 2),
        "bl": (0, h // 2, w // 2, h),
        "br": (w // 2, h // 2, w, h),
    }
    img.crop(boxes[quadrant]).save(out_path)


def pick_quadrant_for_person(label_path):
    """Le o label YOLO e retorna o quadrante (tl/tr/bl/br) onde o centro da
    primeira instancia de Person cai, ou None se nao houver Person."""
    if not label_path.exists():
        return None
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split()
        if int(parts[0]) != PERSON_CLASS_ID_DETECTION:
            continue
        cx, cy = float(parts[1]), float(parts[2])
        return ("t" if cy < 0.5 else "b") + ("l" if cx < 0.5 else "r")
    return None


FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CROPS_DIR.mkdir(parents=True, exist_ok=True)

detector = YOLO(str(DETECTION_WEIGHTS))
segmenter = YOLO(str(SEGMENTATION_WEIGHTS))

images = []
for label_path in sorted(CSS_LABELS_DIR.glob("*.txt")):
    quadrant = pick_quadrant_for_person(label_path)
    if quadrant is None:
        continue
    img_path = CSS_IMAGES_DIR / (label_path.stem + ".jpg")
    if not img_path.exists():
        continue
    out_path = CROPS_DIR / f"{label_path.stem}_{quadrant}.jpg"
    crop_quadrant(img_path, quadrant, out_path)
    images.append(out_path)
    if len(images) >= NUM_EXAMPLES:
        break
print(f"Imagens selecionadas (contem Person): {len(images)}")

for idx, img_path in enumerate(images, start=1):
    det_result = detector.predict(source=str(img_path), conf=0.25, verbose=False)[0]
    seg_result = segmenter.predict(source=str(img_path), conf=0.25, verbose=False)[0]

    det_img = det_result.plot()
    seg_img = seg_result.plot()

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(Image.open(img_path))
    axes[0].set_title("Imagem original")
    axes[0].axis("off")

    axes[1].imshow(det_img[:, :, ::-1])
    axes[1].set_title("Detector Fase 2 (caixas, classe Person)")
    axes[1].axis("off")

    axes[2].imshow(seg_img[:, :, ::-1])
    axes[2].set_title("Segmentador Fase 3 (mascaras, classe person)")
    axes[2].axis("off")

    plt.tight_layout()
    out_path = FIGURES_DIR / f"boxes_vs_masks_example_{idx}.png"
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"Comparacao salva: {out_path}")


# Fase 4 — Avaliação no Conjunto de Teste

Avaliação rigorosa dos dois modelos no conjunto de teste (nunca visto em treino/validação): mAP, IoU, precisão/recall, matriz de confusão, e análise de erros comentada. Requer os pesos treinados em `02_detection.ipynb` e `03_segmentation.ipynb`.

## 11. Avaliação no conjunto de teste

`model.val(split="test")` para os dois modelos: mAP@0.5, mAP@0.5:0.95, precisão, recall, matriz de confusão, e IoU médio (detector).

In [ ]:
import json

from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
import torch

DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
DETECTION_DATA_YAML = ROOT / "data" / "raw" / "construction-site-safety" / "data.yaml"
DETECTION_LABELS_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "labels"
DETECTION_TEST_LIST = ROOT / "data" / "splits" / "construction-site-safety" / "test.txt"

SEGMENTATION_WEIGHTS = ROOT / "models" / "segmentation" / "coco_person_yolov8n_seg_baseline" / "weights" / "best.pt"
SEGMENTATION_DATA_YAML = ROOT / "data" / "raw" / "coco-person" / "data.yaml"

RESULTS_DIR = ROOT / "reports" / "test-evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONF_THRESHOLD = 0.25
IOU_MATCH_THRESHOLD = 0.5


def summarize_val_metrics(results):
    return {
        "precision": float(results.box.mp),
        "recall": float(results.box.mr),
        "mAP50": float(results.box.map50),
        "mAP50-95": float(results.box.map),
    }


def compute_mean_iou_detection():
    """Roda o detector no conjunto de teste, casa cada predicao com a caixa
    de mesma classe com maior IoU no ground-truth, e retorna o IoU medio dos
    pares casados (TP) com confianca >= CONF_THRESHOLD."""
    model = YOLO(str(DETECTION_WEIGHTS))
    test_images = [
        ROOT / "data" / "raw" / "construction-site-safety" / "images" / name
        for name in DETECTION_TEST_LIST.read_text(encoding="utf-8").splitlines()
        if name.strip()
    ]

    ious = []
    for img_path in test_images:
        result = model.predict(source=str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
        pred_boxes = result.boxes.xyxy
        pred_classes = result.boxes.cls

        label_path = DETECTION_LABELS_DIR / (img_path.stem + ".txt")
        if not label_path.exists() or len(pred_boxes) == 0:
            continue

        img_w, img_h = result.orig_shape[1], result.orig_shape[0]
        gt_boxes = []
        gt_classes = []
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = (float(x) for x in parts[1:5])
            gt_boxes.append([
                (cx - bw / 2) * img_w, (cy - bh / 2) * img_h,
                (cx + bw / 2) * img_w, (cy + bh / 2) * img_h,
            ])
            gt_classes.append(cls_id)

        if not gt_boxes:
            continue

        gt_boxes_t = torch.tensor(gt_boxes)
        gt_classes_t = torch.tensor(gt_classes)
        iou_matrix = box_iou(pred_boxes, gt_boxes_t)

        for i in range(len(pred_boxes)):
            same_class = gt_classes_t == pred_classes[i].item()
            if not same_class.any():
                continue
            best_iou = iou_matrix[i][same_class].max().item()
            if best_iou >= IOU_MATCH_THRESHOLD:
                ious.append(best_iou)

    return sum(ious) / len(ious) if ious else 0.0, len(ious)


summary = {}

print("=== Avaliando detector (Fase 2) no conjunto de TESTE ===")
det_model = YOLO(str(DETECTION_WEIGHTS))
det_results = det_model.val(
    data=str(DETECTION_DATA_YAML), split="test",
    project=str(RESULTS_DIR), name="detection_test", exist_ok=True,
)
summary["detection"] = summarize_val_metrics(det_results)

print("\n=== Calculando IoU medio (TP) do detector no conjunto de TESTE ===")
mean_iou, n_matched = compute_mean_iou_detection()
summary["detection"]["mean_iou_tp"] = mean_iou
summary["detection"]["n_matched_boxes"] = n_matched
print(f"IoU medio (predicoes casadas, conf>={CONF_THRESHOLD}): {mean_iou:.4f} (n={n_matched})")

print("\n=== Avaliando segmentador (Fase 3) no conjunto de TESTE ===")
seg_model = YOLO(str(SEGMENTATION_WEIGHTS))
seg_results = seg_model.val(
    data=str(SEGMENTATION_DATA_YAML), split="test",
    project=str(RESULTS_DIR), name="segmentation_test", exist_ok=True,
)
summary["segmentation"] = {
    "box_precision": float(seg_results.box.mp),
    "box_recall": float(seg_results.box.mr),
    "box_mAP50": float(seg_results.box.map50),
    "box_mAP50-95": float(seg_results.box.map),
    "mask_precision": float(seg_results.seg.mp),
    "mask_recall": float(seg_results.seg.mr),
    "mask_mAP50": float(seg_results.seg.map50),
    "mask_mAP50-95": float(seg_results.seg.map),
}

summary_path = RESULTS_DIR / "summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"\nResumo salvo em: {summary_path}")
print(json.dumps(summary, indent=2))


## 12. Análise de erros

Seleciona exemplos de falso positivo/negativo do detector no teste, por maior discrepância de contagem por classe (critério objetivo, não escolha visual).

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
IMAGES_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "images"
LABELS_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "labels"
TEST_LIST = ROOT / "data" / "splits" / "construction-site-safety" / "test.txt"
FIGURES_DIR = ROOT / "reports" / "figures"

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]
CONF_THRESHOLD = 0.25
N_EXAMPLES_EACH = 2


def read_gt_classes(label_path):
    if not label_path.exists():
        return Counter()
    counts = Counter()
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            counts[int(line.split()[0])] += 1
    return counts


FIGURES_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(str(DETECTION_WEIGHTS))

test_names = [n.strip() for n in TEST_LIST.read_text(encoding="utf-8").splitlines() if n.strip()]

fn_candidates = []
fp_candidates = []

for name in test_names:
    img_path = IMAGES_DIR / name
    label_path = LABELS_DIR / (Path(name).stem + ".txt")

    gt_counts = read_gt_classes(label_path)
    result = model.predict(source=str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
    pred_counts = Counter(int(c) for c in result.boxes.cls.tolist())

    missing = gt_counts - pred_counts
    extra = pred_counts - gt_counts

    if missing:
        fn_candidates.append((sum(missing.values()), name, missing))
    if extra:
        fp_candidates.append((sum(extra.values()), name, extra))

fn_candidates.sort(key=lambda x: -x[0])
fp_candidates.sort(key=lambda x: -x[0])

examples = []
for score, name, missing in fn_candidates[:N_EXAMPLES_EACH]:
    classes_str = ", ".join(f"{CLASS_NAMES[c]} x{n}" for c, n in missing.items())
    examples.append(("Falso Negativo", name, classes_str, score))
for score, name, extra in fp_candidates[:N_EXAMPLES_EACH]:
    classes_str = ", ".join(f"{CLASS_NAMES[c]} x{n}" for c, n in extra.items())
    examples.append(("Falso Positivo", name, classes_str, score))

for idx, (kind, name, classes_str, score) in enumerate(examples, start=1):
    img_path = IMAGES_DIR / name
    pred_img = model.predict(source=str(img_path), conf=CONF_THRESHOLD, verbose=False)[0].plot()

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(Image.open(img_path))
    axes[0].set_title(f"Original ({name})")
    axes[0].axis("off")

    axes[1].imshow(pred_img[:, :, ::-1])
    axes[1].set_title(f"Predição do modelo (conf>={CONF_THRESHOLD})")
    axes[1].axis("off")

    fig.suptitle(f"{kind}: {classes_str}", fontsize=12)
    plt.tight_layout()
    out_path = FIGURES_DIR / f"erro_{idx}_{kind.lower().replace(' ', '_')}.png"
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"{kind} salvo: {out_path} — {classes_str}")


## Resultados esperados

Ver `docs/relatorio-tecnico.md` seção 4.2-5 para os números e a análise de erros completa obtidos no desenvolvimento original (mAP@0.5 detector = 0.547, IoU médio = 0.793).

# Fase 4 — Inferência em Vídeo

Concatena os clipes de vídeo do cenário e roda os dois modelos treinados sobre o resultado, salvando as saídas anotadas.

**Antes de rodar:** faça upload dos 3 clipes originais (ver `video/input/README.md` no repositório para as fontes no Pexels) para `PROJECT_DIR/video/input/` no Google Drive.

## 10. Concatenar o vídeo final

Concatena os 3 clipes em um único vídeo ≥30s, padronizando resolução (1280×720, letterbox para os clipes em retrato) e taxa de quadros (25fps).

In [ ]:
import cv2
import numpy as np

INPUT_DIR = ROOT / "video" / "input"
OUTPUT_PATH = INPUT_DIR / "video_final_canteiro_obra.mp4"

# Ordem de concatenacao: paisagem primeiro (mais parecido com o dataset de
# treino), depois os dois clipes em retrato. Fontes/licenca em
# video/input/README.md (Pexels License).
SOURCE_ORDER = [
    "19832492-hd_1280_720_25fps.mp4",
    "14626383_720_1280_30fps.mp4",
    "15518317_720_1280_60fps.mp4",
]

TARGET_W, TARGET_H = 1280, 720
TARGET_FPS = 25.0


def letterbox(frame, target_w, target_h):
    h, w = frame.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off : y_off + new_h, x_off : x_off + new_w] = resized
    return canvas


missing = [f for f in SOURCE_ORDER if not (INPUT_DIR / f).exists()]
if missing:
    print("Faca upload destes clipes para", INPUT_DIR, "antes de continuar:")
    for f in missing:
        print(" -", f)
    print("Fontes: video/input/README.md no repositorio (Pexels).")
else:
    writer = cv2.VideoWriter(
        str(OUTPUT_PATH), cv2.VideoWriter_fourcc(*"mp4v"), TARGET_FPS, (TARGET_W, TARGET_H),
    )

    total_frames_written = 0
    for filename in SOURCE_ORDER:
        src_path = INPUT_DIR / filename
        cap = cv2.VideoCapture(str(src_path))
        src_fps = cap.get(cv2.CAP_PROP_FPS) or TARGET_FPS
        src_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        duration = src_frame_count / src_fps
        n_target_frames = int(duration * TARGET_FPS)
        src_indices = [int(i * src_fps / TARGET_FPS) for i in range(n_target_frames)]

        frames_written_this_clip = 0
        idx_to_frame = {}
        current_idx = -1
        for target_idx in src_indices:
            if target_idx not in idx_to_frame:
                while current_idx < target_idx:
                    ok, frame = cap.read()
                    current_idx += 1
                    if not ok:
                        break
                if ok:
                    idx_to_frame[target_idx] = letterbox(frame, TARGET_W, TARGET_H)
            if target_idx in idx_to_frame:
                writer.write(idx_to_frame[target_idx])
                frames_written_this_clip += 1

        cap.release()
        total_frames_written += frames_written_this_clip
        print(f"{filename}: {frames_written_this_clip} frames escritos "
              f"({frames_written_this_clip / TARGET_FPS:.1f}s)")

    writer.release()
    total_duration = total_frames_written / TARGET_FPS
    print(f"\nVideo final: {OUTPUT_PATH}")
    print(f"Duracao total: {total_duration:.1f}s ({total_frames_written} frames a {TARGET_FPS}fps)")


## 13. Inferência em vídeo

Roda o detector e o segmentador sobre o vídeo final, salvando as saídas anotadas em `video/output/`.

In [ ]:
from ultralytics import YOLO

VIDEO_INPUT = ROOT / "video" / "input" / "video_final_canteiro_obra.mp4"
VIDEO_OUTPUT_DIR = ROOT / "video" / "output"

DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
SEGMENTATION_WEIGHTS = ROOT / "models" / "segmentation" / "coco_person_yolov8n_seg_baseline" / "weights" / "best.pt"

CONF_THRESHOLD = 0.25


def run_inference(weights_path, run_name):
    model = YOLO(str(weights_path))
    model.predict(
        source=str(VIDEO_INPUT), conf=CONF_THRESHOLD, save=True,
        project=str(VIDEO_OUTPUT_DIR), name=run_name, exist_ok=True, verbose=False,
    )
    print(f"Inferencia concluida: {run_name}")


print("=== Inferencia do detector (EPI) no video ===")
run_inference(DETECTION_WEIGHTS, "deteccao_epi")

print("\n=== Inferencia do segmentador (pessoas) no video ===")
run_inference(SEGMENTATION_WEIGHTS, "segmentacao_pessoas")


## Resultado

Os vídeos anotados ficam em `PROJECT_DIR/video/output/deteccao_epi/` e `.../segmentacao_pessoas/` — baixe do Drive para assistir ou usar no vídeo-pitch.

# Aplicação Prática — Relatório de Conformidade de EPI

Transforma as detecções de violação de EPI do vídeo em um relatório de auditoria automática de segurança (eventos com timestamp, duração e confiança). Requer o vídeo final e os pesos do detector (`05_inference_video.ipynb` e `02_detection.ipynb`).

## 14. Detectar violações e gerar o relatório

Roda o detector quadro a quadro, agrupa detecções consecutivas de `NO-Hardhat`, `NO-Mask` e `NO-Safety Vest` em eventos, e salva os dados brutos em Parquet.

In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO

VIDEO_INPUT = ROOT / "video" / "input" / "video_final_canteiro_obra.mp4"
DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
REPORTS_DIR = ROOT / "reports"

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]
VIOLATION_CLASSES = {"NO-Hardhat", "NO-Mask", "NO-Safety Vest"}
CONF_THRESHOLD = 0.25
MAX_GAP_SECONDS = 0.5


def get_video_fps(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps


def collect_frame_detections(fps):
    model = YOLO(str(DETECTION_WEIGHTS))
    detections = []
    frame_idx = 0
    for result in model.predict(source=str(VIDEO_INPUT), conf=CONF_THRESHOLD, stream=True, verbose=False):
        timestamp = frame_idx / fps
        for cls_id, conf in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):
            class_name = CLASS_NAMES[int(cls_id)]
            if class_name in VIOLATION_CLASSES:
                detections.append((timestamp, class_name, conf))
        frame_idx += 1
    return detections


def group_into_events(detections):
    by_class = {}
    for ts, cls, conf in detections:
        by_class.setdefault(cls, []).append((ts, conf))

    events = []
    for cls, items in by_class.items():
        items.sort(key=lambda x: x[0])
        current_start = items[0][0]
        current_end = items[0][0]
        current_confs = [items[0][1]]

        for ts, conf in items[1:]:
            if ts - current_end <= MAX_GAP_SECONDS:
                current_end = ts
                current_confs.append(conf)
            else:
                events.append((cls, current_start, current_end, sum(current_confs) / len(current_confs)))
                current_start = ts
                current_end = ts
                current_confs = [conf]

        events.append((cls, current_start, current_end, sum(current_confs) / len(current_confs)))

    events.sort(key=lambda e: e[1])
    return events


def write_parquet(events, out_path):
    df = pd.DataFrame([
        {
            "classe": cls, "inicio_s": round(start, 2), "fim_s": round(end, 2),
            "duracao_s": round(end - start, 2), "confianca_media": round(conf, 3),
        }
        for cls, start, end, conf in events
    ])
    df.to_parquet(out_path, engine="pyarrow", index=False)
    return df


fps = get_video_fps(VIDEO_INPUT)
print(f"FPS do vídeo: {fps}")

print("Rodando detector quadro a quadro (stream=True)...")
detections = collect_frame_detections(fps)
print(f"Detecções de violação capturadas: {len(detections)}")

events = group_into_events(detections)
print(f"Eventos agregados: {len(events)}")

cap = cv2.VideoCapture(str(VIDEO_INPUT))
video_duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
cap.release()

parquet_path = REPORTS_DIR / "violacoes-epi.parquet"
df = write_parquet(events, parquet_path)
print(f"Parquet salvo: {parquet_path}")
df


## Resumo por classe

In [ ]:
# Resumo por classe (a partir do DataFrame gerado acima)
resumo = df.groupby("classe").agg(
    eventos=("classe", "count"),
    tempo_total_s=("duracao_s", "sum"),
).reset_index()
resumo["pct_do_video"] = (resumo["tempo_total_s"] / video_duration * 100).round(1)
resumo


## Aplicação prática

Este relatório é uma auditoria automática de segurança: em vez de assistir ao vídeo inteiro, um responsável de segurança do trabalho consulta diretamente os eventos acima para saber exatamente quando cada EPI esteve ausente.

Ver `docs/relatorio-tecnico.md` seção 7 para os resultados completos obtidos no desenvolvimento original (ex. NO-Safety Vest presente em 62.8% do vídeo analisado).